# Notebook 04 - SVM phân loại cảm xúc

Notebook này dùng TF-IDF vectorizer đã fit ở Notebook 03, huấn luyện LinearSVC và so sánh với majority-class baseline.

## 0. Khởi tạo môi trường

In [ ]:
from pathlib import Path
import os
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIG_DIR = Path("results/figures")
CSV_DIR = Path("results/csv")
MODEL_DIR = Path("results/models")
for path in [FIG_DIR, CSV_DIR, MODEL_DIR, Path("datasets/cleaned")]:
    path.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore")

## 4.1 Chuẩn bị dữ liệu

In [ ]:
import joblib
import pandas as pd
import numpy as np
from IPython.display import display

from src.classification import (
    analyze_errors,
    cross_validate_svm,
    evaluate_model,
    get_baseline_f1,
    plot_confusion_matrix,
    prediction_results_dataframe,
    train_svm,
)

train_clean = pd.read_csv("datasets/cleaned/train_clean.csv", encoding="utf-8-sig")
test_clean = pd.read_csv("datasets/cleaned/test_clean.csv", encoding="utf-8-sig")
vectorizer = joblib.load("results/models/tfidf_vectorizer.pkl")

X_train = vectorizer.transform(train_clean["comment"].fillna(""))
X_test = vectorizer.transform(test_clean["comment"].fillna(""))
y_train = train_clean["sentiment"]
y_test = test_clean["sentiment"]

label_dist = y_train.value_counts().to_frame("count")
label_dist["percentage"] = (y_train.value_counts(normalize=True) * 100).round(2)
display(label_dist)
print("Nhận xét: Dùng class_weight=balanced để giảm thiên lệch về lớp có nhiều mẫu hơn; F1 Macro được ưu tiên vì đánh giá đều cho cả ba lớp.")

## 4.2 Baseline để so sánh

In [ ]:
baseline_f1 = get_baseline_f1(X_train, y_train, X_test, y_test)
print(f"Baseline F1 Macro: {baseline_f1:.4f}")
print("Nhận xét: Đây là mốc tối thiểu; mô hình học máy cần vượt baseline rõ ràng mới chứng minh được có học từ nội dung bình luận.")

## 4.3 Train LinearSVC

In [ ]:
svm_model = train_svm(X_train, y_train, C=1.0)
print("Đã train LinearSVC(C=1.0, max_iter=2000, random_state=42, class_weight=balanced).")

## 4.4 Cross-validation 5-fold

In [ ]:
cv_scores = cross_validate_svm(X_train, y_train, cv=5)
print(f"Nhận xét: F1 Macro trung bình = {cv_scores.mean():.4f}, độ lệch chuẩn = {cv_scores.std():.4f}.")
if cv_scores.std() < 0.03:
    print("Model khá ổn định giữa các fold.")
else:
    print("Model dao động giữa các fold; nên kiểm tra mất cân bằng lớp hoặc dữ liệu nhiễu.")

## 4.5 Đánh giá trên test set

In [ ]:
metrics = evaluate_model(svm_model, X_test, y_test, ["Positive", "Negative", "Neutral"])
improvement = (metrics["f1_macro"] - baseline_f1) / baseline_f1 * 100 if baseline_f1 else np.nan
print(f"Baseline F1 Macro    : {baseline_f1:.4f}")
print(f"SVM F1 Macro (test)  : {metrics['f1_macro']:.4f} ({improvement:+.2f}% so với baseline)")
print(f"SVM F1 Weighted      : {metrics['f1_weighted']:.4f}")
print(f"SVM Accuracy         : {metrics['accuracy']:.4f}")
print(f"SVM Precision (macro): {metrics['precision_macro']:.4f}")
print(f"SVM Recall (macro)   : {metrics['recall_macro']:.4f}")
print("Nhận xét: Nếu F1 Macro tăng mạnh so với baseline, mô hình đã học được tín hiệu cảm xúc từ văn bản.")

## 4.6 Confusion Matrix

In [ ]:
labels = ["Positive", "Negative", "Neutral"]
cm = plot_confusion_matrix(svm_model, X_test, y_test, labels, FIG_DIR / "confusion_matrix.png")
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
display(cm_df)

correct_by_label = pd.Series(np.diag(cm), index=labels).sort_values(ascending=False)
print(f"Nhãn dự đoán đúng nhiều nhất: {correct_by_label.index[0]} ({correct_by_label.iloc[0]} mẫu đúng).")

cm_errors = cm_df.copy()
for label in labels:
    cm_errors.loc[label, label] = 0
max_true = cm_errors.stack().idxmax()
max_error = cm_errors.stack().max()
print(f"Cặp nhầm nhiều nhất: nhãn thật {max_true[0]} bị đoán thành {max_true[1]} ({max_error} mẫu).")
print("Nhận xét: Neutral thường dễ bị nhầm vì nhiều câu trung tính có từ tích cực nhẹ hoặc tiêu cực nhẹ.")

## 4.7 Phân tích lỗi

In [ ]:
errors = analyze_errors(svm_model, X_test, y_test, test_clean, n=5)
display(errors)
for _, row in errors.iterrows():
    print("Bình luận:", row["comment"])
    print("Nhãn thật:", row["y_true"])
    print("SVM đoán :", row["y_pred"])
    print("Phân tích: Câu có thể chứa từ khóa mơ hồ, phủ định, hoặc nhiều aspect trái chiều nên TF-IDF khó nắm trọn ngữ nghĩa.\n")

## 4.8 Lưu mô hình và kết quả

In [ ]:
svm_results = prediction_results_dataframe(svm_model, X_test, y_test, test_clean)
svm_results.to_csv(CSV_DIR / "svm_results.csv", index=False, encoding="utf-8-sig")
joblib.dump(svm_model, MODEL_DIR / "svm_model.pkl")

metrics_summary = pd.DataFrame([{**metrics, "baseline_f1_macro": baseline_f1, "cv_mean": cv_scores.mean(), "cv_std": cv_scores.std()}])
metrics_summary.to_csv(CSV_DIR / "svm_metrics_summary.csv", index=False, encoding="utf-8-sig")
print("Đã lưu SVM model, kết quả dự đoán và confusion matrix.")